In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd


# --- Chemins compatibles Notebook ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "retail_customers_COMPLETE_CATEGORICAL.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("RAW exists?:", RAW_PATH.exists())



def safe_to_datetime(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def safe_to_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def churn_to_binary(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float).round().clip(0, 1).astype(int)

    m = {
        "1": 1, "true": 1, "yes": 1, "y": 1, "oui": 1, "parti": 1, "churn": 1,
        "0": 0, "false": 0, "no": 0, "n": 0, "non": 0, "fidèle": 0, "loyal": 0,
    }
    x = s.astype(str).str.strip().str.lower().map(m)
    num = pd.to_numeric(s, errors="coerce")
    x = x.fillna(num)
    return x.fillna(0).astype(float).round().clip(0, 1).astype(int)


def iqr_outlier_rate(series: pd.Series) -> float:
    """Return outlier percentage with IQR method for numeric series."""
    x = safe_to_numeric(series).dropna()
    if x.empty:
        return 0.0
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    out = ((x < low) | (x > high)).mean() * 100
    return float(out)


def main() -> None:
    if not RAW_PATH.exists():
        raise FileNotFoundError(
            f"Dataset introuvable: {RAW_PATH}\n"
            f"Place le fichier CSV dans data/raw/."
        )

    df = pd.read_csv(RAW_PATH)
    df.columns = [str(c).strip() for c in df.columns]

    print("\n==================== 1) STRUCTURE GLOBALE ====================")
    print(f"Dimensions: {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print("\nTypes de colonnes:")
    print(df.dtypes.value_counts())

    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]

    print(f"\nColonnes numériques ({len(num_cols)}): {num_cols[:12]}{' ...' if len(num_cols)>12 else ''}")
    print(f"Colonnes catégorielles/texte ({len(cat_cols)}): {cat_cols[:12]}{' ...' if len(cat_cols)>12 else ''}")

    print("\n==================== 2) QUALITÉ DES DONNÉES ====================")

    # Missing values
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    print("\nTop colonnes avec valeurs manquantes (%):")
    print(missing_pct.head(15))

    # Duplicates
    dup_rows = df.duplicated().sum()
    print(f"\nLignes dupliquées exactes: {dup_rows} ({dup_rows/len(df)*100:.2f}%)")

    # Constant columns
    nunique = df.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()
    print(f"\nColonnes constantes (inutiles): {constant_cols if constant_cols else 'Aucune'}")

    # High cardinality categoricals
    high_card = {}
    for c in cat_cols:
        card = df[c].nunique(dropna=True)
        if card >= 20:
            high_card[c] = card
    print("\nColonnes catégorielles à forte cardinalité (>=20):")
    print(high_card if high_card else "Aucune")

    # Outlier rates on numeric columns
    outlier_stats = []
    for c in num_cols:
        rate = iqr_outlier_rate(df[c])
        outlier_stats.append((c, rate))
    outlier_df = pd.DataFrame(outlier_stats, columns=["column", "outlier_pct"]).sort_values(
        "outlier_pct", ascending=False
    )

    print("\nTop colonnes avec outliers (IQR, %):")
    print(outlier_df.head(10).to_string(index=False))

    # Domain checks (si colonnes présentes)
    print("\n==================== 3) CONTRÔLES MÉTIER ====================")
    domain_rows = []

    if "Age" in df.columns:
        age = safe_to_numeric(df["Age"])
        bad_age = ((age < 15) | (age > 100)).mean() * 100
        domain_rows.append(("Age hors [15,100]", bad_age))

    if "Satisfaction" in df.columns:
        sat = safe_to_numeric(df["Satisfaction"])
        bad_sat = ((sat < 1) | (sat > 5)).mean() * 100
        domain_rows.append(("Satisfaction hors [1,5]", bad_sat))

    if "SupportTickets" in df.columns:
        st = safe_to_numeric(df["SupportTickets"])
        bad_st = (st < 0).mean() * 100
        domain_rows.append(("SupportTickets < 0", bad_st))

    # Date parsing quality
    date_candidates = [c for c in df.columns if c.lower() in {"registdate", "registrationdate", "registerdate", "signupdate"}]
    if date_candidates:
        dcol = date_candidates[0]
        parsed = safe_to_datetime(df[dcol])
        bad_date = parsed.isna().mean() * 100
        domain_rows.append((f"{dcol} non parseable", bad_date))

    if domain_rows:
        domain_df = pd.DataFrame(domain_rows, columns=["check", "bad_pct"]).sort_values("bad_pct", ascending=False)
        print(domain_df.to_string(index=False))
    else:
        domain_df = pd.DataFrame(columns=["check", "bad_pct"])
        print("Aucun contrôle métier spécifique détecté (colonnes absentes).")

    print("\n==================== 4) CIBLE CHURN ====================")
    if "Churn" in df.columns:
        y = churn_to_binary(df["Churn"])
        churn_dist = y.value_counts(normalize=True).sort_index() * 100
        print("Distribution Churn (%):")
        print(churn_dist.rename(index={0: "Non churn (0)", 1: "Churn (1)"}))
        if len(churn_dist) == 2:
            ratio = churn_dist.max() / max(churn_dist.min(), 1e-9)
            print(f"Ratio de déséquilibre approx: {ratio:.2f}:1")
    else:
        print("Colonne 'Churn' absente.")

    print("\n==================== 5) EXPORT RAPPORT ====================")

    # Build report table
    rows = []
    for c in df.columns:
        rows.append(
            {
                "column": c,
                "dtype": str(df[c].dtype),
                "n_unique": int(df[c].nunique(dropna=True)),
                "missing_pct": float(df[c].isna().mean() * 100),
                "sample_values": ", ".join(map(str, df[c].dropna().astype(str).head(3).tolist())),
            }
        )

    col_report = pd.DataFrame(rows).sort_values(["missing_pct", "n_unique"], ascending=[False, False])
    col_report.to_csv(REPORTS_DIR / "exploration_columns_report.csv", index=False)
    outlier_df.to_csv(REPORTS_DIR / "exploration_outliers_report.csv", index=False)

    if not domain_df.empty:
        domain_df.to_csv(REPORTS_DIR / "exploration_domain_checks.csv", index=False)

    print(f"Rapports générés dans: {REPORTS_DIR}")
    print("- exploration_columns_report.csv")
    print("- exploration_outliers_report.csv")
    if not domain_df.empty:
        print("- exploration_domain_checks.csv")

    print("\n✅ Exploration terminée.")


if __name__ == "__main__":
    main()


: 